<a href="https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hasnaincoder1/Flyrankrepo1/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** Refresh / Content Opportunity Scoring.

**Question:** out of a client's existing content library, which pages should an editor review for a refresh first this week?

**Decision it supports:** an SEO/content editor with limited review hours picks pages off a ranked queue rather than guessing or reviewing oldest-first. A false negative (missing a page that's genuinely declining) is the more expensive mistake — lost search visibility is harder to win back the longer it's ignored than the cost of one editor-hour spent checking a page that turned out fine.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release used:** the FlyRank ML Internship starter export, `data/raw/content_refresh_anonymized.csv` — 30,000 anonymized content pages across 32 clients, each row a trailing-90-day snapshot with a 30-day-vs-prior-30-day trend comparison.

**Honest scope note:** ML-04 set up the warehouse data contract for the full ~79M-row Hugging Face release, but I did not have Hugging Face network access or a stored `HF_TOKEN` in the environment I built this in, so this paper's model and numbers are built entirely on the local 30K-row starter export, not the full warehouse. That's a real limitation, named again in Section 5.

**Excluded, and why:** `trend_direction` and `trend_pct` — these define the label itself (`is_declining_label = trend_direction == "down"`), so they're never features. `impressions_last_30d` / `impressions_prev_30d` are also excluded even though not explicitly forbidden — the label is a deterministic threshold on exactly those two numbers, so including them would let a model reconstruct the label almost perfectly rather than learn a real pattern (confirmed directly in Section 3's leakage audit). No client names, URLs, or raw queries appear anywhere in this notebook or the paper — `content_id`/`client_id` are already anonymized hashes in the source export.


In [ ]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} pages across {df['client_id'].nunique()} clients")
print(f"base decline rate: {df['is_declining_label'].mean():.1%}")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label:** `is_declining_label = (trend_direction == "down")` — a rule-derived proxy on measured impression change, not a hand-tagged outcome.

**Features:** 18 numeric (search volume, competition, CPC, word/char count, log-transformed 90-day impressions/clicks/sessions/AI-sessions, days with impressions/sessions, content age, days since last update, CTR, avg position, engagement/scroll rate, AI traffic %) + 7 categorical (competition level, content type, main intent, age tier, freshness tier, word-count tier, impression tier).

**Baseline (ML-07):** a hand-written rule, `declining_and_stale_with_demand` — flagged 5,270 of 30,000 pages. Auditing it honestly in ML-08 revealed it was **circular**: it multiplied the label directly into its own score, so it scored a trivial precision@50 = 1.0 by construction. I built a **fair baseline** instead — same staleness+demand logic, label term removed — for an honest comparison.

**Validation design:** grouped split by `client_id` (75/25, `GroupShuffleSplit`), confirmed zero client overlap between train and test — the model is scored only on 8 clients it never trained on.

**Leakage checks (ML-09):** adding the excluded `trend_pct` back as a feature sent AUC from 0.616 to 0.987 — a clean confession the label-derived column had to be excluded. Adding the raw `impressions_last_30d`/`prev_30d` pair pushed AUC to 0.830, confirming those needed excluding too.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
import pandas as pd
results = pd.DataFrame([
    {"model": "Fair baseline (staleness + demand, no label term)", "precision@50": 0.30, "precision@100": 0.30, "auc": None},
    {"model": "Base rate (random)", "precision@50": 0.517, "precision@100": 0.517, "auc": 0.50},
    {"model": "Random Forest (300 trees, depth 8)", "precision@50": 0.56, "precision@100": 0.50, "auc": 0.60},
    {"model": "Logistic Regression", "precision@50": 0.76, "precision@100": 0.72, "auc": 0.62},
])
results

## 5. Limitations

*What this work cannot claim.*

- **Small sample, not the full warehouse.** 30,000 pages / 32 clients from the starter export — not the ~79M-row production release. Results here are directional evidence the approach is worth extending, not a finding validated at production scale.
- **Observational, not causal.** Nothing here says refreshing a flagged page *causes* recovery — only that current signals match the pattern associated with decline in this sample. The source paper's own Freshness Multiplier finding (audited in ML-09) faces the identical caveat: refreshed pages are editor-selected, not randomly assigned.
- **A rule-derived proxy label**, not an observed real-world outcome — `is_declining_label` is a threshold on measured impression change, defined by the pipeline, not tagged by a human reviewer.
- **Validated on 8 held-out clients.** A meaningfully larger held-out set would strengthen confidence in how well this generalizes to a client the model has never seen.
- **~1 in 4 false positive rate** at the top of the queue (precision@50 = 0.76) — a ranking tool for human triage, not a certainty machine.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

From ML-10's action playbook, scored on the same held-out clients:


In [ ]:
import json
with open("work/outputs/w07_playbook_metrics.json") as f:
    playbook = json.load(f)
print(f"queue size: {playbook['queue_size']:,} | flagged refresh_review: {playbook['n_flagged_refresh_review']:,}")
print()
print("Reason codes:", playbook["reason_code_counts"])
print()
print("Archetypes:", playbook["archetype_counts"])
print()
print(f"click-equivalent value on flagged pages: ${playbook['total_click_equivalent_value_flagged']:,.0f}")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Three charts, already generated and committed under `work/figures/`:
- `capstone_model_vs_baseline.png` — precision@50/100 across baseline, base rate, and both models
- `capstone_feature_importance.png` — permutation importance, what the model actually leans on
- `capstone_leakage_confession.png` — the AUC jump when label-derived features are added back in
- `w07_reason_code_distribution.png` — the action queue's reason-code breakdown


In [ ]:
import os
for f in ["capstone_model_vs_baseline.png", "capstone_feature_importance.png",
          "capstone_leakage_confession.png", "w07_reason_code_distribution.png"]:
    path = f"work/figures/{f}"
    print(f"{path}: {'OK, ' + str(os.path.getsize(path)) + ' bytes' if os.path.exists(path) else 'MISSING'}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
